<a href="https://colab.research.google.com/github/Harmandeepkaur370/project6th/blob/main/Copy_of_project6th.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🏥 AI HOSPITAL MEDICAL INTELLIGENCE SYSTEM
# WITH PRECAUTIONS AFTER PREDICTION
# ============================================================

!pip install -q gradio scikit-learn xgboost reportlab pandas numpy matplotlib

import gradio as gr
import pandas as pd
import numpy as np
import os
import datetime
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet


# ============================================================
# UI STYLE
# ============================================================

css="""
body{
background: linear-gradient(135deg,#e3f2fd,#f1f8ff);
}

.title{
text-align:center;
font-size:36px;
font-weight:bold;
color:#1565C0;
}

.subtitle{
text-align:center;
font-size:18px;
color:gray;
margin-bottom:20px;
}

.card{
background:white;
padding:20px;
border-radius:15px;
box-shadow:0 4px 15px rgba(0,0,0,0.1);
}

.login-box{
width:350px;
margin:auto;
padding:30px;
border-radius:15px;
background:white;
box-shadow:0px 6px 20px rgba(0,0,0,0.15);
}
"""

# ============================================================
# DATABASE FILES
# ============================================================

users_file="users_db.csv"
history_file="patients_history.csv"


# ============================================================
# USER DATABASE
# ============================================================

def load_users():
    if os.path.exists(users_file):
        return pd.read_csv(users_file)
    return pd.DataFrame(columns=["username","password"])


def signup(username,password):

    df=load_users()

    if username in df["username"].values:
        return "User already exists"

    new=pd.DataFrame([[username,password]],columns=["username","password"])
    df=pd.concat([df,new],ignore_index=True)

    df.to_csv(users_file,index=False)

    return "Signup successful. Please login."


def login(username,password):

    df=load_users()

    if ((df["username"]==username)&(df["password"]==password)).any():
        return gr.update(visible=False),gr.update(visible=True),"Login Successful"

    return gr.update(visible=True),gr.update(visible=False),"Invalid Login"


def logout():
    return gr.update(visible=True),gr.update(visible=False)


# ============================================================
# DATASETS
# ============================================================

diabetes_url="https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

cols=["Pregnancies","Glucose","BloodPressure","SkinThickness",
      "Insulin","BMI","DiabetesPedigreeFunction","Age","Outcome"]

diabetes_df=pd.read_csv(diabetes_url,names=cols)

heart_url="https://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
heart_df=pd.read_csv(heart_url)

for col in heart_df.columns:
    if heart_df[col].dtype=="object":
        heart_df[col]=heart_df[col].astype("category").cat.codes


# ============================================================
# MODEL TRAINING
# ============================================================

X_heart=heart_df.drop("target",axis=1)
y_heart=heart_df["target"]

X_diabetes=diabetes_df.drop("Outcome",axis=1)
y_diabetes=diabetes_df["Outcome"]

Xh_train,Xh_test,yh_train,yh_test=train_test_split(X_heart,y_heart,test_size=0.2,random_state=42)
Xd_train,Xd_test,yd_train,yd_test=train_test_split(X_diabetes,y_diabetes,test_size=0.2,random_state=42)

heart_model=XGBClassifier(n_estimators=200)
heart_model.fit(Xh_train,yh_train)

diabetes_model=RandomForestClassifier(n_estimators=200)
diabetes_model.fit(Xd_train,yd_train)

heart_acc=accuracy_score(yh_test,heart_model.predict(Xh_test))
diabetes_acc=accuracy_score(yd_test,diabetes_model.predict(Xd_test))


# ============================================================
# HISTORY
# ============================================================

def save_history(name,age,disease,risk):

    time=datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    new=pd.DataFrame([[name,age,disease,risk,time]],
    columns=["Name","Age","Disease","Risk","Time"])

    if os.path.exists(history_file):
        new.to_csv(history_file,mode="a",header=False,index=False)
    else:
        new.to_csv(history_file,index=False)


def load_history():

    if os.path.exists(history_file):
        return pd.read_csv(history_file)

    return pd.DataFrame(columns=["Name","Age","Disease","Risk","Time"])


# ============================================================
# HOSPITAL ANALYTICS
# ============================================================

def hospital_stats():

    df=load_history()

    if len(df)==0:
        return "No records yet."

    heart=len(df[df["Disease"]=="Heart Disease"])
    diabetes=len(df[df["Disease"]=="Diabetes"])

    return f"""
### Hospital Analytics

Total Patients: **{len(df)}**

Heart Predictions: **{heart}**

Diabetes Predictions: **{diabetes}**
"""


# ============================================================
# LIFESTYLE MODIFIER
# ============================================================

def lifestyle_modifier(prob,smoking,exercise,diet,family):

    if smoking==1: prob+=0.08
    if exercise==0: prob+=0.07
    if diet==0: prob+=0.05
    if family==1: prob+=0.1

    return min(prob,1)


# ============================================================
# RISK CHART
# ============================================================

def risk_chart(prob):

    percent=int(prob*100)

    fig,ax=plt.subplots()
    ax.barh(["Risk"],[percent])

    ax.set_xlim(0,100)
    ax.set_xlabel("Risk %")
    ax.set_title("Disease Risk")

    return fig


# ============================================================
# PRECAUTIONS
# ============================================================

def precautions(disease,risk):

    if disease=="Heart Disease":

        if risk=="LOW":
            tips=[
            "Maintain regular physical activity",
            "Eat low-fat diet",
            "Avoid smoking",
            "Maintain healthy weight"
            ]

        elif risk=="MODERATE":
            tips=[
            "Monitor blood pressure",
            "Reduce cholesterol intake",
            "Increase exercise",
            "Consult doctor for screening"
            ]

        else:
            tips=[
            "Immediate cardiologist consultation",
            "Strict cholesterol control",
            "Stop smoking immediately",
            "Follow medical treatment"
            ]


    if disease=="Diabetes":

        if risk=="LOW":
            tips=[
            "Maintain balanced diet",
            "Exercise regularly",
            "Monitor weight",
            "Annual glucose test"
            ]

        elif risk=="MODERATE":
            tips=[
            "Reduce sugar intake",
            "Increase fiber foods",
            "Daily exercise",
            "Check blood glucose regularly"
            ]

        else:
            tips=[
            "Consult endocrinologist",
            "Monitor glucose daily",
            "Strict diabetic diet",
            "Possible medication"
            ]

    html="<div class='card'><h3>Recommended Precautions</h3><ul>"

    for t in tips:
        html+=f"<li>{t}</li>"

    html+="</ul></div>"

    return html


# ============================================================
# REPORT
# ============================================================

def generate_report(prob,disease,name,age):

    percent=int(prob*100)

    if percent<30:
        risk="LOW"
        color="green"
    elif percent<70:
        risk="MODERATE"
        color="orange"
    else:
        risk="HIGH"
        color="red"

    html=f"""
<div class='card'>
<h2>{disease} Risk Assessment</h2>

<b>Patient:</b> {name}<br>
<b>Age:</b> {age}

<br><br>

<div style="height:30px;background:#eee;border-radius:15px">
<div style="width:{percent}%;height:30px;background:{color};
border-radius:15px;text-align:center;color:white;font-weight:bold">

{percent}% Risk

</div>
</div>

<h3 style="color:{color}">{risk} RISK</h3>

</div>
"""

    html+=precautions(disease,risk)

    return html,risk


# ============================================================
# PDF REPORT
# ============================================================

def create_pdf(name,age,disease,risk):

    file="medical_report.pdf"

    doc=SimpleDocTemplate(file)

    styles=getSampleStyleSheet()

    elements=[]

    elements.append(Paragraph("AI Hospital Medical Report",styles["Heading1"]))
    elements.append(Spacer(1,20))
    elements.append(Paragraph(f"Patient: {name}",styles["Normal"]))
    elements.append(Paragraph(f"Age: {age}",styles["Normal"]))
    elements.append(Paragraph(f"Disease: {disease}",styles["Normal"]))
    elements.append(Paragraph(f"Risk Level: {risk}",styles["Normal"]))

    doc.build(elements)

    return file


# ============================================================
# HEART PREDICTION
# ============================================================

def heart_predict(name,age,sex,cp,bp,chol,fbs,thalach,smoking,exercise,diet,family):

    arr=[[age,sex,cp,bp,chol,fbs,thalach]]

    arr=pd.DataFrame(arr,
    columns=["age","sex","cp","trestbps","chol","fbs","thalach"])

    arr=arr.reindex(columns=X_heart.columns,fill_value=0)

    prob=heart_model.predict_proba(arr)[0][1]

    prob=lifestyle_modifier(prob,smoking,exercise,diet,family)

    html,risk=generate_report(prob,"Heart Disease",name,age)

    chart=risk_chart(prob)

    save_history(name,age,"Heart Disease",risk)

    pdf=create_pdf(name,age,"Heart Disease",risk)

    return html,pdf,chart,load_history()


# ============================================================
# DIABETES PREDICTION
# ============================================================

def diabetes_predict(name,age,preg,glucose,bp,bmi,insulin,smoking,exercise,diet,family):

    arr=[[preg,glucose,bp,0,insulin,bmi,0,age]]

    prob=diabetes_model.predict_proba(arr)[0][1]

    prob=lifestyle_modifier(prob,smoking,exercise,diet,family)

    html,risk=generate_report(prob,"Diabetes",name,age)

    chart=risk_chart(prob)

    save_history(name,age,"Diabetes",risk)

    pdf=create_pdf(name,age,"Diabetes",risk)

    return html,pdf,chart,load_history()


# ============================================================
# SYMPTOM CHECKER
# ============================================================

def symptom_checker(symptoms):

    symptoms=set(symptoms)

    diseases=[]

    if {"chest pain","shortness of breath"} & symptoms:
        diseases.append("Possible Heart Disease")

    if {"frequent urination","weight loss"} & symptoms:
        diseases.append("Possible Diabetes")

    if len(diseases)==0:
        diseases.append("No major disease detected")

    result="<br>".join(diseases)

    return f"<div class='card'><h3>AI Symptom Analysis</h3>{result}</div>"


# ============================================================
# UI
# ============================================================

with gr.Blocks(css=css,theme=gr.themes.Soft()) as app:

    gr.Markdown("""
<div class='title'>🏥 AI Hospital Medical Intelligence System</div>
<div class='subtitle'>Heart Disease • Diabetes Prediction • Symptom Analysis</div>
""")

    # LOGIN PAGE
    with gr.Column(elem_classes="login-box",visible=True) as login_page:

        gr.Markdown("### Login")

        user=gr.Textbox(label="Username")
        pwd=gr.Textbox(label="Password",type="password")

        login_btn=gr.Button("Login")

        gr.Markdown("### Signup")

        new_user=gr.Textbox(label="New Username")
        new_pass=gr.Textbox(label="New Password",type="password")

        signup_btn=gr.Button("Signup")

        msg=gr.Textbox(label="Status")

    # DASHBOARD
    with gr.Column(visible=False) as dashboard:

        logout_btn=gr.Button("Logout")

        gr.Markdown(f"""
### Model Accuracy

Heart Model: **{heart_acc*100:.2f}%**

Diabetes Model: **{diabetes_acc*100:.2f}%**
""")

        with gr.Row():

            with gr.Column():

                name=gr.Textbox(label="Patient Name")
                age=gr.Slider(20,80,label="Age")
                sex=gr.Radio([0,1],label="Gender")

                smoking=gr.Radio([0,1],label="Smoking")
                exercise=gr.Radio([0,1],label="Exercise")
                diet=gr.Radio([0,1],label="Healthy Diet")
                family=gr.Radio([0,1],label="Family History")

            with gr.Column():

                with gr.Tabs():

                    with gr.Tab("Heart Disease"):

                        cp=gr.Slider(0,3,label="Chest Pain")
                        bp=gr.Slider(80,200,label="Blood Pressure")
                        chol=gr.Slider(100,400,label="Cholesterol")
                        fbs=gr.Radio([0,1],label="Fasting Sugar")
                        thalach=gr.Slider(70,210,label="Max Heart Rate")

                        heart_btn=gr.Button("Predict Heart Risk")

                        heart_out=gr.HTML()
                        heart_pdf=gr.File()
                        heart_chart=gr.Plot()

                    with gr.Tab("Diabetes"):

                        preg=gr.Slider(0,15,label="Pregnancies")
                        glucose=gr.Slider(70,200,label="Glucose")
                        bp2=gr.Slider(60,150,label="Blood Pressure")
                        bmi=gr.Slider(18,45,label="BMI")
                        insulin=gr.Slider(0,300,label="Insulin")

                        dia_btn=gr.Button("Predict Diabetes Risk")

                        dia_out=gr.HTML()
                        dia_pdf=gr.File()
                        dia_chart=gr.Plot()

                    with gr.Tab("Symptom Checker"):

                        symptoms=gr.CheckboxGroup([
                        "chest pain","shortness of breath","fatigue",
                        "frequent urination","dizziness","headache",
                        "weight loss"])

                        symptom_btn=gr.Button("Analyze Symptoms")

                        symptom_output=gr.HTML()

        gr.Markdown("### Patient History")

        history_table=gr.Dataframe(value=load_history)

        gr.Markdown("### Hospital Analytics")

        stats_box=gr.Markdown(hospital_stats)

        heart_btn.click(
        heart_predict,
        [name,age,sex,cp,bp,chol,fbs,thalach,smoking,exercise,diet,family],
        [heart_out,heart_pdf,heart_chart,history_table]
        )

        dia_btn.click(
        diabetes_predict,
        [name,age,preg,glucose,bp2,bmi,insulin,smoking,exercise,diet,family],
        [dia_out,dia_pdf,dia_chart,history_table]
        )

        symptom_btn.click(symptom_checker,symptoms,symptom_output)

    login_btn.click(login,[user,pwd],[login_page,dashboard,msg])
    signup_btn.click(signup,[new_user,new_pass],msg)
    logout_btn.click(logout,[],[login_page,dashboard])

app.launch(share=True)

/tmp/ipykernel_159/1999493981.py:455: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css,theme=gr.themes.Soft()) as app:
/tmp/ipykernel_159/1999493981.py:455: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css,theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40cca311b26b0ebc05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
